# Compare peri-trigger firing histogram max times

This notebook peri-trigger firing histograms between spike-types, within groups.

First, for each cell, collects the specified triggers and calculates the firing histogram around the given time window around each trigger for each spike-type separately. After smoothing and averaging the histograms, the time of the maximal firing frequency is detected for each spike type.

Finally, a paired hypothesis test is performed within a group of cells to test the difference between spike types.

## Initialization - don't change

In [ ]:
import sys
sys.path.append('..')

from workflows.run_util.io import collect_paths_from_table
from param_classes.workflows.peritrigger_analysis import PeriTriggerAnalysisWorkflowParams

from workflows.peritrigger_analysis import run_analysis

params = PeriTriggerAnalysisWorkflowParams()

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
params.data.raw_data.path='C:\\Datasets\\raap\\invivo'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
params.data.spike_features.path = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\SpikeFeatures'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
params.input = {
    'mode': 'table',
    'table_path': 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
params.output.path = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\PeriTriggerAnalysis'

## Analysis parameters

First, you can set the peritrigger analysis parameters.
 - `trigger`: marker name that you want to use for triggering
 - `spike_types`: name of the spike types that you want to compare
 - `trigger_condition`: if you leave empty (`[]`), all triggers will be used. If you put values in the list, only those triggers will be used for the analysis where that spike type was present

In [ ]:
params.analysis.trigger = 'delta_end'
params.analysis.spike_types = ['AIS-AP', 'RA-AP']
params.analysis.trigger_condition = ['RA-AP']
params.analysis.smooth_size = 5

Here you can set the histogram parameters.
 - `data_unit`: time unit that you want to use. You have to give the following parameters in this unit too. Possible values: `'s'` or `'ms'`
 - `bin_range`: range around the trigger to analyze
 - `bin_size`: size of the bin
 - `value_unit`: unit of the histogram, possible values: `'count'`, `'freq'`, `'prob'`

In [ ]:
params.analysis.histogram.data_unit = 'ms'
params.analysis.histogram.bin_range = (-100, 400)
params.analysis.histogram.bin_size = 1
params.analysis.histogram.value_unit = 'freq'

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(params.data, params.input)
print(f"{len(input_files)} files selected for analysis.")

Perform individual analysis and save the results.

In [ ]:
trigger_results = run_analysis(input_files, params)

In [ ]:
test_sample = trigger_results[0]
dataframe = test_sample.data
centers = dataframe['centers']
test_hist = dataframe.iloc[:, 7]
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
fig, ax = plt.subplots(1, 2, figsize=(20, 5), sharey=True)
for i, spike_type in enumerate(params.analysis.spike_types):
    flag = dataframe['spike_type'] == spike_type
    spike_hists = dataframe[flag].iloc[:, 2:].values
    smoothed_hists = gaussian_filter1d(spike_hists, sigma=5, axis=0)
    ax[i].plot(
        centers[flag],
        np.mean(spike_hists, axis=1),
        label=spike_type
    )
    ax[i].plot(
        centers[flag],
        np.mean(smoothed_hists, axis=1),
        label=spike_type
    )
plt.show()
